# Entregável 9 — Seleção de Atributos (Feature Selection)

**Disciplina:** Aquisição de Biossinais  
**Equipe:** José Lessa & Matheus Rocha  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL (PhysioNet)  

---

## Objetivo

No Entregável 8 mitigamos a dimensionalidade usando PCA, o que converte os dados em "Componentes Matemáticos de Variância" abstratos. A técnica desta fase (**Feature Selection**) é a *alternativa interpretável*: identificaremos o Subconjunto Ideal das features **fisiológicas originais** descartando colunas inteiras do banco que não ajudam o algoritmo ou causam Overfitting.

### Conteúdo abordado pela exigência da Pipeline:
1. **Método Filtro:** Mutual Information (Ganho de Informação Teórico).
2. **Método Embedded:** Importance Scores extraídos direto dos Nodos de um *Random Forest Classifier*.
3. **Método Wrapper (Validação Mestra):** RFECV (Recursive Feature Elimination com Stratified Cross-validation) para garantir **Estabilidade Absoluta da Seleção**.

## 1. Configuração e Imports

In [ ]:
!pip install scikit-learn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.feature_selection import SequentialFeatureSelector, RFE, RFECV, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

# Configurações de plot originais da disciplina
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True

BASE_DIR = Path('..')
FIG_DIR = Path('figuras')
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Configuração pronta.')

## 2. Carregamento do Dataset de Extração (Engenheirado E.7)

Observem: Para extrair métricas de dependência fisiológica, importamos o CSV de Features Puras Z-Score (do E7), **e não do PCA (E8)**, dado que no PCA as colunas originais deixaram de existir.

In [ ]:
csv_path = BASE_DIR / 'entregavel-7' / 'dataset_features_engenheirado.csv'
df = pd.read_csv(csv_path)

cols_meta = ['ecg_id', 'derivacao', 'janela_instancia', 'target_is_norm']
cols_feat = [c for c in df.columns if c not in cols_meta]

X = df[cols_feat]
y = df['target_is_norm']

print(f"-> Carregadas {X.shape[0]} amostras com {len(cols_feat)} Features candidatas para a Peneira.")

## 3. Abordagem FILTRO (Filter Method) 
Aplica estatística direta Feature-Target sem treinar modelos pesados. Usaremos o robustíssimo **Mutual Information (Info Gain)**, que ao contrário de correlações lineares (Pearson), flagra dependências fortíssimas em formatos parabólicos (ex: U-shape features relativas à taqui/bradicardia).

In [ ]:
print('Calculando Entropia Mútua... (Information Gain)')
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=cols_feat).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=mi_series.head(15).values, y=mi_series.head(15).index, palette='viridis')
plt.title('TOP 15 Features por Information Gain (Mutual Information Classif)', fontweight='bold')
plt.xlabel('Mutual Information Score (Bits de certeza fornecidos sobre o Diagnóstico)')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(FIG_DIR / 'filter_mutual_info.png')
plt.show()

## 4. Abordagem EMBEDDED (Model Based)
Treina o poderoso Random Forest e abre o cérebro das árvores pra saber quais métricas matemáticas ele mais usou como nó de decisão (Folha/Raiz) para gerar o "sim ou não" de maneira ótima (Gini Importance).

In [ ]:
print('Treinando Random Forest para extração do Gini Impurity Importance...')
rf = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X, y)

rf_importances = pd.Series(rf.feature_importances_, index=cols_feat).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=rf_importances.head(15).values, y=rf_importances.head(15).index, palette='magma')
plt.title('TOP 15 Features por Gini Importance (Random Forest Algorítmico)', fontweight='bold')
plt.xlabel('Grau de Dependência Relativa no Gini-Impurity (Soma = 1.0)')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(FIG_DIR / 'embedded_rf_importance.png')
plt.show()

## 5. Abordagem WRAPPER e Validação Obrigatória (Estabilidade da Seleção)

As técnicas anteriores rankeam métricas isoladamente. O problema é que 5 Variáveis Ótimas independentemente podem não jogar bem em equipe. 

O **Wrapper RFECV (Recursive Feature Elimination with Cross Validation)** treina o modelo, capta o menos importante, e joga fora iterativamente, repetindo até a **acurácia** começar a cair. Isso roda dentro de um sistema robusto K-Fold para atestar **Estabilidade Inabalável** do número de atributos escolhido de acordo co o Edital do Pipeline.

In [ ]:
print('=' * 80)
print('INICIANDO PENEIRA WRAPPER RFECV COM STRATIFIED 5-FOLD CORSS_VALIDATION')
print('=' * 80)

# Devido ao alto tempo convolucional N! em computadores locais, usaremos a RF leve (n_estimators 50)
rf_base = RandomForestClassifier(n_estimators=50, max_depth=7, random_state=42, n_jobs=-1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rfecv = RFECV(estimator=rf_base, step=1, cv=cv, scoring='accuracy', n_jobs=-1)
rfecv.fit(X, y)

print(f"-> Número Fantástico/Ótimo calculado de features: {rfecv.n_features_}")

# 5.1 VALIDAÇÃO - Gráfico de Estabilidade de CV
plt.figure(figsize=(10, 6))

cv_results = pd.DataFrame(rfecv.cv_results_)
mean_test_scores = cv_results['mean_test_score']
std_test_scores = cv_results['std_test_score'] / np.sqrt(cv.n_splits)
n_features_range = range(1, len(mean_test_scores) + 1)

plt.plot(n_features_range, mean_test_scores, marker='o', color='purple', label='Acurácia Média')
plt.fill_between(n_features_range, 
                 mean_test_scores - std_test_scores, 
                 mean_test_scores + std_test_scores, color='purple', alpha=0.2, label='Standart Error (Estabilidade CV)')

plt.axvline(rfecv.n_features_, color='red', linestyle='--', label=f'Corte Ideal = {rfecv.n_features_} features')

plt.title('Validação da Estabilidade da Seleção (RFECV Wrapper)', fontweight='bold', fontsize=14)
plt.xlabel('Número Estrito de Features Empregadas no Conjunto de Treino')
plt.ylabel('Desempenho Estabilizado do Cross-Validation')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'wrapper_estabilidade_cv.png')
plt.show()

# Coletar o nome das Vencedoras
suporte_boolean = rfecv.support_
vencedoras = np.array(cols_feat)[suporte_boolean]
print(f"Lista Bruta das Vencedoras Exclusivas Selecionadas: {list(vencedoras)}")

## 6. Salvar Tabela do Subconjunto Final Selecionado

Através da peneira mais implacável do mercado de I.A (Wrapper RFECV), as n_features finalistas comporão a tabela oficial e otimizada (leve para inferências instantâneas) para o treinamento definitivo na classificação!

In [ ]:
# Exportar tabela das top finalistas isoladas como arquivo
df_final_selected = df[cols_meta + list(vencedoras)].copy()

# Imprimir o "Report" exigido no pipeline
report_df = pd.DataFrame({
    'Feature Aprovada (RFECV)': vencedoras,
    'Importance Score (Gini)': rf_importances.loc[list(vencedoras)].values,
    'Information Gain Mútuo': mi_series.loc[list(vencedoras)].values 
})
report_df.sort_values(by='Importance Score (Gini)', ascending=False, inplace=True)

print('=' * 80)
print("TABELA RASTREADORA - SUBCONJUNTO IDEAL MAIS ESTÁVEL DO PACIENTE")
print('=' * 80)
display(report_df)

# Salvar
output_csv = FIG_DIR.parent / 'dataset_features_selecionadas_puras.csv'
df_final_selected.to_csv(output_csv, index=False)

print(f"\nArquivo FINAL salvo em Ouro! -> {output_csv}")
print(f"O Dataset para o Entregável 10 (Modelos) passou de absurdas {len(cols_feat)} Features Puras originais para as incríveis e estáveis {rfecv.n_features_} features fisiológicas exatas.")